## Hi we will Implement the Neural Networks I have learnt till now in this Notebook

In [402]:
print("Hello World")

Hello World


Now lets go ahead and build a Neural Network. Yay!

# Importing Libraries

In [403]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import math
import zipfile
import os
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

First Lets Define Our **Neural Network**

# Defining Our Model

In [404]:
class NN_Network(nn.Module):
    def __init__(self,input_size,hidden_size):
        super(NN_Network,self).__init__()
        self.fc1 = nn.Linear(input_size,hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size,1)
    def forward(self,x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

Now That Our Model is Defined , Lets set the dimensions and Hyperparameter for our model
Our Model currently has dimentions:
input data size,
hidden units,
And it has parameters like:
$\alpha$ which is the learning rate


In [405]:
input_size = 20
hidden_size = 8
output_size = 2
learning_rate = 0.01

In this Notebook we will train our model to predict whether a team will be able to chase down a score or not in a cricket match in **IPL**, Lets load the data

# Data Processing

In [406]:
path = 'archive.zip'
with zipfile.ZipFile(path,'r') as z:
    z.extractall("archive")
data_path = 'archive/'

Now lets look at the files

In [407]:
os.listdir(data_path)

['IPL_ball_by_ball_updated.csv',
 'deliveries_updated_ipl_upto_2025.csv',
 'deliveries_updated_mens_ipl.csv',
 'matches_updated_ipl_upto_2025.csv',
 'matches_updated_mens_ipl.csv']

Here we have different csv files which contain our data, First we need to process this data and load it carefully into our training and test data, for that we can use the pandas package which has been already imported above

In [408]:
ball_by_ball = pd.read_csv("archive/IPL_ball_by_ball_updated.csv")
matches = pd.read_csv("archive/matches_updated_ipl_upto_2025.csv")
ball_by_ball.shape
ball_by_ball.columns
matches.columns

Index(['season', 'venue', 'event', 'winner_runs', 'umpire2', 'toss_winner',
       'date', 'neutralvenue', 'umpire1', 'city', 'reserve_umpire', 'winner',
       'eliminator', 'date1', 'method', 'team1', 'toss_decision', 'gender',
       'team2', 'balls_per_over', 'winner_wickets', 'tv_umpire',
       'player_of_match', 'match_referee', 'outcome', 'date2', 'match_number',
       'matchId'],
      dtype='object')

We will use important columns only that will useful for training our model, which include the following(**REMEMBER OUR MODEL WILL BE TRAINED SUCH THAT IT PREDICTS THE OUTCOME AFTER $10$ OVERS ARE BOWLED**):
"season",
"venue",
"batting_team",
"bowling_team", 
"ball",
"runs_off_bat",
"extras",
"wicket_type",
"player_dismissed"
From Matches data we will use the following columns-
"matchId",
"winner",
"team1",
"team2",
"toss_winner",
"toss_decision",
"venue",
"season",
"balls_per_over"


First Lets Select only the 2nd innings data, since we want to predict the outcome of a chase and chase happens in the 2nd innings

In [409]:
ball_by_ball = ball_by_ball.sort_values(
    ['match_id','innings','ball'])

ball_by_ball['ball_in_innings'] = (
    ball_by_ball
    .groupby(['match_id','innings'])
    .cumcount() + 1
)
bb = ball_by_ball[
    (ball_by_ball['innings'] == 2) &
    (ball_by_ball['ball_in_innings'] <= 60)
]


Now Lets group the relevant columns

In [410]:
X = bb.groupby('match_id').agg(
    runs_scored_10 = ('runs_off_bat','sum'),
    extras_10     = ('extras','sum'),
    wickets_lost_10 = ('player_dismissed','count'),
    balls_played  = ('ball_in_innings','count')
).reset_index()

X['runs_scored_10'] += X['extras_10']
X['wickets_remaining'] = 10 - X['wickets_lost_10']


In [411]:
X = X.merge(
    matches[['matchId','venue','season','team2','team1']],
    left_on='match_id',
    right_on='matchId',
    how='inner'
)

Lets look at our data Now, we can there see are a few columns we dont need that is matchID columns, because they are not features, we just wanted them to get our data by match, lets get rid of them, but before we drop matchid lets get our **Y** labels

In [412]:
y_df = matches[['matchId','winner']]
data = X.merge(
    y_df,
    left_on='match_id',
    right_on='matchId',
    how='inner'
)
#these are our labels
Y = (data['winner'] == data['team1']).astype(int)

In [413]:
X = X.drop(columns=['match_id','matchId'])

In [414]:
X['chasing_team'] = X['team1']   # chasing team
X['bowling_team'] = X['team2']

In [415]:
X = X.drop(columns=['balls_played','team1','team2'])

Some of our columns in X have values that are strings, our NN won't learn well if we dont make those strings into numeric form, for that we will use scikit

In [416]:
num_cols = ['runs_scored_10','extras_10','wickets_lost_10','wickets_remaining']
cat_cols = ['venue','season','chasing_team','bowling_team']

In [417]:
preprocessor = ColumnTransformer(transformers=[('num',StandardScaler(),num_cols),('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)])

In [418]:
X_processed = preprocessor.fit_transform(X)

**Now Lets Split our Data into Training and Test Sets**

In [419]:
X_train,X_test,y_train,y_test = train_test_split(X_processed,Y,test_size=0.1,random_state=42,stratify=Y)

Now in PyTorch we need to use tensors to train our model, so we need to convert our data into tensors

In [420]:
X_train_t = torch.tensor(X_train.toarray(),dtype = torch.float32)
X_test_t = torch.tensor(X_test.toarray(),dtype = torch.float32)
Y_train_t = torch.tensor(y_train.values, dtype = torch.float32).unsqueeze(1)
Y_test_t = torch.tensor(y_test.values, dtype = torch.float32).unsqueeze(1)

Now lets Initialize our Model that we have created,Why use **BCEWithLogitsLoss** here ?

Our task is binary classification.

This loss:

combines sigmoid + binary cross-entropy,

is numerically stable

and avoids vanishing gradients 

We will use the Standard Adam Optimizer

In [421]:
model = NN_Network(input_size=X_train_t.shape[1],hidden_size=64)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# TRAINING

Now lets Build a Training Loop and run it.Yay ! (Thats refreshing)

In [422]:
epochs = 100

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits,Y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 100 ==0:
        print(f"Epoch-{epoch+1}, Loss {loss.item():.4f}")
    

Epoch-100, Loss 0.0219


# TESTING

In [423]:
model.eval()

with torch.no_grad():
    logits = model(X_test_t)
    probs = torch.sigmoid(logits)
    preds = (probs > 0.7).int()

accuracy = (preds == Y_test_t.int()).float().mean()
print("Test accuracy:", accuracy.item())


Test accuracy: 0.5145630836486816


**This Model Gives a accuracy close to 52-55 % , which is good because in cricket there are many variables that can determine the match outcome which happen in last 5 overs, which our model has never seen. Unless we do feature engineering or encorporate more data like player performances that season, venue chase history ,team chase history which can add another 5-6 % but beyond that our model would plateau. So, this was a good learning excercise on how to get a dataset and build a neural network and test it's realistic performance.**